### Project Solution: Goals 1 and 2

In this notebook we will build the solution step by step.

The project asks us to create:

* a `Polygon` class
* a finite sequence of polygons
* tests for the functionality

We will use plain `assert` statements for our tests, just like we would do while learning or prototyping.


### Goal 1

We need to create a `Polygon` class.

Each polygon is a regular convex polygon centered at `(0, 0)`.

The initializer needs:

* `n`: the number of vertices, minimum `3`
* `R`: the circumradius, meaning the distance from the center to every vertex

All the other values can be computed from `n` and `R`.


For a regular polygon with `n` vertices and circumradius `R`, we will use these formulas:

* interior angle: `(n - 2) * 180 / n`
* side length: `2 * R * sin(pi / n)`
* apothem: `R * cos(pi / n)`
* perimeter: `n * side_length`
* area: `1 / 2 * perimeter * apothem`

The area formula can also be written as:

```python
n / 2 * side_length * apothem
```


Let's start by importing `math`, since we will need `sin`, `cos`, `pi`, and `isclose`.


In [1]:
import math


Let's create a first version of our `Polygon` class.

This version already includes the core computed properties.


In [2]:
class Polygon:
    def __init__(self, n, R):
        self._n = n
        self._R = R

    def __repr__(self):
        return f'Polygon(n={self._n}, R={self._R})'

    @property
    def count_vertices(self):
        return self._n

    @property
    def count_edges(self):
        return self._n

    @property
    def count_sides(self):
        return self._n

    @property
    def circumradius(self):
        return self._R

    @property
    def interior_angle(self):
        return (self._n - 2) * 180 / self._n

    @property
    def side_length(self):
        return 2 * self._R * math.sin(math.pi / self._n)

    @property
    def edge_length(self):
        return self.side_length

    @property
    def apothem(self):
        return self._R * math.cos(math.pi / self._n)

    @property
    def perimeter(self):
        return self._n * self.side_length

    @property
    def area(self):
        return self.perimeter * self.apothem / 2

    @property
    def area_to_perimeter_ratio(self):
        return self.area / self.perimeter


Let's test the representation and the simple properties first.

Notice that we are using `assert`.

If an assertion is true, nothing is displayed.  
If an assertion is false, Python raises an `AssertionError`.


In [3]:
def test_polygon_basic_properties():
    n = 3
    R = 1
    p = Polygon(n, R)

    assert str(p) == 'Polygon(n=3, R=1)', f'actual: {str(p)}'
    assert p.count_vertices == n, f'actual: {p.count_vertices}, expected: {n}'
    assert p.count_edges == n, f'actual: {p.count_edges}, expected: {n}'
    assert p.count_sides == n, f'actual: {p.count_sides}, expected: {n}'
    assert p.circumradius == R, f'actual: {p.circumradius}, expected: {R}'
    assert p.interior_angle == 60, f'actual: {p.interior_angle}, expected: 60'


In [4]:
test_polygon_basic_properties()


So far, so good.

Next we will test some computed values.

For a square with circumradius `1`:

* side length should be `sqrt(2)`
* perimeter should be `4 * sqrt(2)`
* apothem should be `sqrt(2) / 2`
* area should be `2`

When testing floats, we should not use exact equality.  
Instead, we will use `math.isclose`.


In [5]:
def test_polygon_square_values():
    abs_tol = 0.001
    rel_tol = 0.001

    p = Polygon(4, 1)

    assert p.interior_angle == 90, f'actual: {p.interior_angle}, expected: 90'

    assert math.isclose(
        p.side_length, math.sqrt(2),
        rel_tol=rel_tol, abs_tol=abs_tol
    ), f'actual: {p.side_length}, expected: {math.sqrt(2)}'

    assert math.isclose(
        p.perimeter, 4 * math.sqrt(2),
        rel_tol=rel_tol, abs_tol=abs_tol
    ), f'actual: {p.perimeter}, expected: {4 * math.sqrt(2)}'

    assert math.isclose(
        p.apothem, math.sqrt(2) / 2,
        rel_tol=rel_tol, abs_tol=abs_tol
    ), f'actual: {p.apothem}, expected: {math.sqrt(2) / 2}'

    assert math.isclose(
        p.area, 2,
        rel_tol=rel_tol, abs_tol=abs_tol
    ), f'actual: {p.area}, expected: 2'


In [6]:
test_polygon_square_values()


Let's test a few more known values.

For `n = 6`, `R = 2`:

* side length is approximately `2`
* apothem is approximately `1.73205`
* area is approximately `10.3923`
* perimeter is approximately `12`
* interior angle is `120`

For `n = 12`, `R = 3`:

* side length is approximately `1.55291`
* apothem is approximately `2.89778`
* area is approximately `27`
* perimeter is approximately `18.635`
* interior angle is `150`


In [7]:
def test_polygon_more_values():
    abs_tol = 0.001
    rel_tol = 0.001

    p = Polygon(6, 2)

    assert math.isclose(p.side_length, 2, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.apothem, 1.73205, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.area, 10.3923, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.perimeter, 12, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.interior_angle, 120, rel_tol=rel_tol, abs_tol=abs_tol)

    p = Polygon(12, 3)

    assert math.isclose(p.side_length, 1.55291, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.apothem, 2.89778, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.area, 27, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.perimeter, 18.635, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.interior_angle, 150, rel_tol=rel_tol, abs_tol=abs_tol)


In [8]:
test_polygon_more_values()


Now we need to add a few more things:

* validation: `n` must be an integer greater than or equal to `3`
* validation: `R` must be a positive finite number
* equality based on `n` and `R`
* ordering based on `n` only
* immutability

For immutability, we will block attribute changes after initialization.


In [9]:
class Polygon:
    __slots__ = ('_n', '_R', '_locked')

    def __init__(self, n, R):
        if isinstance(n, bool) or not isinstance(n, int):
            raise TypeError('n must be an integer')

        if n < 3:
            raise ValueError('A polygon must have at least 3 vertices.')

        if isinstance(R, bool) or not isinstance(R, (int, float)):
            raise TypeError('R must be a real number')

        if not math.isfinite(R) or R <= 0:
            raise ValueError('R must be a positive finite number.')

        object.__setattr__(self, '_n', n)
        object.__setattr__(self, '_R', R)
        object.__setattr__(self, '_locked', True)

    def __setattr__(self, name, value):
        if getattr(self, '_locked', False):
            raise AttributeError('Polygon objects are immutable.')
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        raise AttributeError('Polygon objects are immutable.')

    def __repr__(self):
        return f'Polygon(n={self._n}, R={self._R})'

    @property
    def count_vertices(self):
        return self._n

    @property
    def num_vertices(self):
        return self._n

    @property
    def count_edges(self):
        return self._n

    @property
    def num_edges(self):
        return self._n

    @property
    def count_sides(self):
        return self._n

    @property
    def circumradius(self):
        return self._R

    @property
    def interior_angle(self):
        return (self._n - 2) * 180 / self._n

    @property
    def side_length(self):
        return 2 * self._R * math.sin(math.pi / self._n)

    @property
    def edge_length(self):
        return self.side_length

    @property
    def apothem(self):
        return self._R * math.cos(math.pi / self._n)

    @property
    def perimeter(self):
        return self._n * self.side_length

    @property
    def area(self):
        return self.perimeter * self.apothem / 2

    @property
    def area_to_perimeter_ratio(self):
        return self.area / self.perimeter

    @property
    def vertex_coordinates(self):
        return tuple(
            (
                self._R * math.cos(2 * math.pi * vertex / self._n),
                self._R * math.sin(2 * math.pi * vertex / self._n)
            )
            for vertex in range(self._n)
        )

    def as_dict(self):
        return {
            'n': self._n,
            'R': self._R,
            'interior_angle': self.interior_angle,
            'side_length': self.side_length,
            'apothem': self.apothem,
            'area': self.area,
            'perimeter': self.perimeter,
            'area_to_perimeter_ratio': self.area_to_perimeter_ratio
        }

    def __eq__(self, other):
        if isinstance(other, Polygon):
            return self.count_vertices == other.count_vertices and self.circumradius == other.circumradius
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, Polygon):
            return self.count_vertices < other.count_vertices
        return NotImplemented

    def __le__(self, other):
        if isinstance(other, Polygon):
            return self.count_vertices <= other.count_vertices
        return NotImplemented

    def __gt__(self, other):
        if isinstance(other, Polygon):
            return self.count_vertices > other.count_vertices
        return NotImplemented

    def __ge__(self, other):
        if isinstance(other, Polygon):
            return self.count_vertices >= other.count_vertices
        return NotImplemented


Let's now test the final version of the `Polygon` class.

We'll test validation first.


In [10]:
def assert_raises(exception_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except exception_type:
        return
    except Exception as ex:
        assert False, f'expected {exception_type.__name__}, but got {type(ex).__name__}'
    else:
        assert False, f'expected {exception_type.__name__}, but no exception was raised'


In [11]:
def test_polygon_validation():
    assert_raises(ValueError, Polygon, 2, 1)
    assert_raises(TypeError, Polygon, 3.5, 1)
    assert_raises(TypeError, Polygon, True, 1)

    assert_raises(ValueError, Polygon, 3, 0)
    assert_raises(ValueError, Polygon, 3, -1)
    assert_raises(ValueError, Polygon, 3, float('inf'))
    assert_raises(ValueError, Polygon, 3, float('nan'))
    assert_raises(TypeError, Polygon, 3, '10')


In [12]:
test_polygon_validation()


Next, let's re-run our previous tests.

Since we redefined the class, this confirms that the final class still has the correct core behavior.


In [13]:
test_polygon_basic_properties()
test_polygon_square_values()
test_polygon_more_values()


Now let's test equality and ordering.

The project requires:

* equality based on number of vertices and circumradius
* ordering based on number of vertices only


In [14]:
def test_polygon_equality_and_ordering():
    p1 = Polygon(3, 10)
    p2 = Polygon(10, 10)
    p3 = Polygon(15, 10)
    p4 = Polygon(15, 100)
    p5 = Polygon(15, 100)

    assert p2 > p1
    assert p2 < p3

    assert p3 != p4
    assert p1 != p4
    assert p4 == p5

    # Ordering uses only the number of vertices.
    assert p3 >= p4
    assert p3 <= p4


In [15]:
test_polygon_equality_and_ordering()


Finally, let's test that a polygon is immutable.

We should not be able to change `_n`, `_R`, or add a new attribute after the object has been created.


In [16]:
def test_polygon_immutability():
    p = Polygon(4, 1)

    assert_raises(AttributeError, setattr, p, '_n', 10)
    assert_raises(AttributeError, setattr, p, '_R', 100)
    assert_raises(AttributeError, setattr, p, 'new_attribute', 'not allowed')
    assert_raises(AttributeError, delattr, p, '_n')


In [17]:
test_polygon_immutability()


Let's also check our small useful extra: vertex coordinates.

For a square with circumradius `1`, the first point starts at angle `0`, so it should be `(1, 0)`.


In [18]:
def test_polygon_vertex_coordinates():
    p = Polygon(4, 1)
    points = p.vertex_coordinates

    assert len(points) == 4

    x, y = points[0]
    assert math.isclose(x, 1, abs_tol=0.001)
    assert math.isclose(y, 0, abs_tol=0.001)

    for x, y in points:
        distance_from_center = math.sqrt(x ** 2 + y ** 2)
        assert math.isclose(distance_from_center, 1, abs_tol=0.001)


In [19]:
test_polygon_vertex_coordinates()


At this point, our `Polygon` class is complete.

Now we can move on to the finite sequence type.


### Goal 2

We need to create a finite sequence of polygons.

The sequence should contain polygons starting with `3` vertices and ending with `m` vertices.

For example:

```python
polygons = Polygons(6, 10)
```

should contain:

```python
Polygon(3, 10)
Polygon(4, 10)
Polygon(5, 10)
Polygon(6, 10)
```

So the length should be:

```python
m - 2
```


The sequence needs to support:

* `len(polygons)`
* positive indexing
* negative indexing
* slicing
* extended slicing
* highest area-to-perimeter ratio polygon

Since all polygons have the same circumradius, the polygon with the highest `area / perimeter` ratio is the polygon with the largest number of vertices.


In [20]:
class Polygons:
    __slots__ = ('_m', '_R', '_locked')

    def __init__(self, m, R):
        if isinstance(m, bool) or not isinstance(m, int):
            raise TypeError('m must be an integer')

        if m < 3:
            raise ValueError('m must be at least 3')

        if isinstance(R, bool) or not isinstance(R, (int, float)):
            raise TypeError('R must be a real number')

        if not math.isfinite(R) or R <= 0:
            raise ValueError('R must be a positive finite number.')

        object.__setattr__(self, '_m', m)
        object.__setattr__(self, '_R', R)
        object.__setattr__(self, '_locked', True)

    def __setattr__(self, name, value):
        if getattr(self, '_locked', False):
            raise AttributeError('Polygons objects are immutable.')
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        raise AttributeError('Polygons objects are immutable.')

    def __repr__(self):
        return f'Polygons(m={self._m}, R={self._R})'

    @property
    def max_vertices(self):
        return self._m

    @property
    def circumradius(self):
        return self._R

    @property
    def length(self):
        return len(self)

    def __len__(self):
        return self._m - 2

    def __getitem__(self, index):
        vertex_numbers = range(3, self._m + 1)

        if isinstance(index, slice):
            return tuple(Polygon(n, self._R) for n in vertex_numbers[index])

        if isinstance(index, bool) or not isinstance(index, int):
            raise TypeError('sequence indices must be integers or slices')

        try:
            n = vertex_numbers[index]
        except IndexError:
            raise IndexError('polygon index out of range')

        return Polygon(n, self._R)

    def __iter__(self):
        for n in range(3, self._m + 1):
            yield Polygon(n, self._R)

    def __contains__(self, polygon):
        return (
            isinstance(polygon, Polygon)
            and polygon.circumradius == self._R
            and 3 <= polygon.count_vertices <= self._m
        )

    @property
    def max_area_perimeter_ratio_polygon(self):
        return self[-1]

    @property
    def most_efficient_polygon(self):
        return self.max_area_perimeter_ratio_polygon

    def ratios(self):
        return tuple(p.area_to_perimeter_ratio for p in self)

    def summary(self):
        return tuple(p.as_dict() for p in self)


Let's test the basic sequence behavior first.


In [21]:
def test_polygons_basic_sequence():
    polygons = Polygons(6, 10)

    assert str(polygons) == 'Polygons(m=6, R=10)', f'actual: {str(polygons)}'
    assert len(polygons) == 4
    assert polygons.length == 4

    assert polygons[0] == Polygon(3, 10)
    assert polygons[1] == Polygon(4, 10)
    assert polygons[2] == Polygon(5, 10)
    assert polygons[3] == Polygon(6, 10)


In [22]:
test_polygons_basic_sequence()


Now let's test negative indexing.


In [23]:
def test_polygons_negative_indexing():
    polygons = Polygons(6, 10)

    assert polygons[-1] == Polygon(6, 10)
    assert polygons[-2] == Polygon(5, 10)
    assert polygons[-3] == Polygon(4, 10)
    assert polygons[-4] == Polygon(3, 10)


In [24]:
test_polygons_negative_indexing()


Now let's test that invalid indexing raises the correct exceptions.


In [25]:
def test_polygons_invalid_indexing():
    polygons = Polygons(6, 10)

    assert_raises(IndexError, polygons.__getitem__, 4)
    assert_raises(IndexError, polygons.__getitem__, -5)
    assert_raises(TypeError, polygons.__getitem__, 1.5)
    assert_raises(TypeError, polygons.__getitem__, True)


In [26]:
test_polygons_invalid_indexing()


Next, let's test slicing.

Slicing should return a tuple of polygons.


In [27]:
def test_polygons_slicing():
    polygons = Polygons(8, 1)

    assert polygons[0:3] == (
        Polygon(3, 1),
        Polygon(4, 1),
        Polygon(5, 1)
    )

    assert polygons[2:] == (
        Polygon(5, 1),
        Polygon(6, 1),
        Polygon(7, 1),
        Polygon(8, 1)
    )

    assert polygons[:2] == (
        Polygon(3, 1),
        Polygon(4, 1)
    )


In [28]:
test_polygons_slicing()


Extended slicing should also work.

That includes things like:

```python
polygons[::2]
polygons[::-1]
```


In [29]:
def test_polygons_extended_slicing():
    polygons = Polygons(10, 1)

    assert polygons[::2] == (
        Polygon(3, 1),
        Polygon(5, 1),
        Polygon(7, 1),
        Polygon(9, 1)
    )

    assert polygons[1:7:2] == (
        Polygon(4, 1),
        Polygon(6, 1),
        Polygon(8, 1)
    )

    assert polygons[::-1] == (
        Polygon(10, 1),
        Polygon(9, 1),
        Polygon(8, 1),
        Polygon(7, 1),
        Polygon(6, 1),
        Polygon(5, 1),
        Polygon(4, 1),
        Polygon(3, 1)
    )


In [30]:
test_polygons_extended_slicing()


Let's also test iteration and membership.


In [31]:
def test_polygons_iteration_and_membership():
    polygons = Polygons(6, 3)

    assert tuple(polygons) == (
        Polygon(3, 3),
        Polygon(4, 3),
        Polygon(5, 3),
        Polygon(6, 3)
    )

    assert Polygon(3, 3) in polygons
    assert Polygon(6, 3) in polygons
    assert Polygon(7, 3) not in polygons
    assert Polygon(6, 4) not in polygons
    assert 'not a polygon' not in polygons


In [32]:
test_polygons_iteration_and_membership()


Now let's test the highest `area / perimeter` ratio.

For a fixed circumradius, this should be the polygon with the largest number of vertices in the sequence.


In [33]:
def test_polygons_best_ratio():
    polygons = Polygons(12, 5)

    expected = max(polygons, key=lambda p: p.area_to_perimeter_ratio)

    assert polygons.max_area_perimeter_ratio_polygon == expected
    assert polygons.max_area_perimeter_ratio_polygon == Polygon(12, 5)
    assert polygons.most_efficient_polygon == Polygon(12, 5)


In [34]:
test_polygons_best_ratio()


Let's test the sequence validation and immutability too.


In [35]:
def test_polygons_validation_and_immutability():
    assert_raises(ValueError, Polygons, 2, 1)
    assert_raises(TypeError, Polygons, 3.5, 1)
    assert_raises(TypeError, Polygons, True, 1)

    assert_raises(ValueError, Polygons, 3, 0)
    assert_raises(ValueError, Polygons, 3, -1)
    assert_raises(ValueError, Polygons, 3, float('inf'))
    assert_raises(ValueError, Polygons, 3, float('nan'))
    assert_raises(TypeError, Polygons, 3, '10')

    polygons = Polygons(8, 2)

    assert_raises(AttributeError, setattr, polygons, '_m', 100)
    assert_raises(AttributeError, setattr, polygons, '_R', 100)
    assert_raises(AttributeError, setattr, polygons, 'new_attribute', 'not allowed')
    assert_raises(AttributeError, delattr, polygons, '_m')


In [36]:
test_polygons_validation_and_immutability()


Finally, let's test the small extra utility methods.

The `summary()` method returns a tuple of dictionaries, one dictionary per polygon.

The `ratios()` method returns all area-to-perimeter ratios.


In [37]:
def test_polygons_extra_utilities():
    polygons = Polygons(5, 1)

    ratios = polygons.ratios()
    summary = polygons.summary()

    assert len(ratios) == len(polygons)
    assert len(summary) == len(polygons)

    assert isinstance(summary[0], dict)
    assert summary[0]['n'] == 3
    assert summary[-1]['n'] == 5

    assert ratios == tuple(p.area_to_perimeter_ratio for p in polygons)


In [38]:
test_polygons_extra_utilities()


Let's run all the tests together.

This gives us one clean cell that can be re-run whenever we make changes.


In [39]:
def run_all_tests():
    test_polygon_basic_properties()
    test_polygon_square_values()
    test_polygon_more_values()
    test_polygon_validation()
    test_polygon_equality_and_ordering()
    test_polygon_immutability()
    test_polygon_vertex_coordinates()

    test_polygons_basic_sequence()
    test_polygons_negative_indexing()
    test_polygons_invalid_indexing()
    test_polygons_slicing()
    test_polygons_extended_slicing()
    test_polygons_iteration_and_membership()
    test_polygons_best_ratio()
    test_polygons_validation_and_immutability()
    test_polygons_extra_utilities()

    print('All tests passed!')


In [40]:
run_all_tests()


All tests passed!


### Example usage

Let's create a sequence of polygons from triangles to octagons with circumradius `3`.


In [41]:
polygons = Polygons(8, 3)

print(polygons)
print('length:', len(polygons))
print('first polygon:', polygons[0])
print('last polygon:', polygons[-1])
print('every other polygon:', polygons[::2])
print('best ratio polygon:', polygons.max_area_perimeter_ratio_polygon)


Polygons(m=8, R=3)
length: 6
first polygon: Polygon(n=3, R=3)
last polygon: Polygon(n=8, R=3)
every other polygon: (Polygon(n=3, R=3), Polygon(n=5, R=3), Polygon(n=7, R=3))
best ratio polygon: Polygon(n=8, R=3)


And let's print a small table manually using formatted strings.


In [42]:
print(f'{"n":>3} {"side":>10} {"apothem":>10} {"area":>10} {"perimeter":>12} {"ratio":>10}')
print('-' * 61)

for p in polygons:
    print(
        f'{p.count_vertices:>3} '
        f'{p.side_length:>10.4f} '
        f'{p.apothem:>10.4f} '
        f'{p.area:>10.4f} '
        f'{p.perimeter:>12.4f} '
        f'{p.area_to_perimeter_ratio:>10.4f}'
    )


  n       side    apothem       area    perimeter      ratio
-------------------------------------------------------------
  3     5.1962     1.5000    11.6913      15.5885     0.7500
  4     4.2426     2.1213    18.0000      16.9706     1.0607
  5     3.5267     2.4271    21.3988      17.6336     1.2135
  6     3.0000     2.5981    23.3827      18.0000     1.2990
  7     2.6033     2.7029    24.6277      18.2231     1.3515
  8     2.2961     2.7716    25.4558      18.3688     1.3858


### Final notes

The solution now satisfies the project requirements:

* `Polygon` objects are immutable
* the polygon sequence is immutable
* all requested polygon properties are implemented
* equality is based on number of vertices and circumradius
* ordering is based on number of vertices
* sequence indexing supports positive indices, negative indices, slicing, and extended slicing
* the sequence supports `len()`
* the sequence provides the polygon with the highest `area / perimeter` ratio
* tests cover the important behavior and edge cases

We also added two useful extras:

* `vertex_coordinates`
* summary and ratio helpers for the sequence
